# PKG network metrics (F3), part 1: monthly local metrics and graph exports for the GPU job

**PNC Treasury Management · DSI · Payment Knowledge Graph** · design: Claude Doc "PKG Network Metrics (F3) — Design
for approval" (2026-11-26/27). Independent of periodicity.

| step | what | where |
|---|---|---|
| §2 local | `pkg_cust_net_monthly`: local, stress and dynamic metrics per scope customer (TM customers and organisations) and month | Spark, this notebook |
| §3 export | per window (3 months ending each month): internal, observed and community (family) graphs + node table, copied to `NET_DIR/w=<month>/` | Spark → shared folder |
| GPU job | `pkg_network_gpu.py` (submit `pkg_network_gpu.sh` with `sbatch`): PageRank, HITS, clustering, components, core number, trophic level, circulation, Infomap communities with stability ensemble, month-to-month matching | GPU node, reads `NET_DIR` |
| part 2 (next) | publish: snapshot tables, community table, flow matrix, backbone, consensus replacement, flow atlas | Spark |

**Run tonight:** start this notebook (Run All), then submit the GPU job. The GPU job waits for each exported window and
processes them in order; both resume after a stop (markers per month / window).

Rules: own accounts (`is_self`) out; counterparty ids → `ent_l2` (lowest per id); intra-family edges (same
`master_pwr_id`) flagged and kept out of exposure, reciprocity, reach and every graph; hubs cut by degree (customers:
≥ `HUB_MULT` × median; entities: `hub_tier` 3 or ≥ `ENT_HUB_DEG` scope customers in the window); people only as
counterparties in local metrics.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
import os
DB = "bdahd01p_dlcdi1_cdi_tm"
REF = "_v5"
T_EDGE_M = f"{DB}.pkg_edge_monthly"
T_CUST   = f"{DB}.pkg_cust_dim{REF}"
T_MAP    = f"{DB}.pkg_cpty_entity_map{REF}"
T_ENT    = f"{DB}.pkg_cpty_entity_dim{REF}"
T_CPM    = f"{DB}.pkg_cpty_metrics_monthly{REF}"
OUT_TABLES = {"local": f"{DB}.pkg_cust_net_monthly"}
NET_DIR = os.path.abspath("../net_work")      # shared folder read by the GPU job (same path on both hosts)
OUT_DIR = "../metrics/network"

MONTHS_BACK = None             # None = every published edge month; a number = only the latest that many
LAST_MONTH  = None             # None = latest published edge month
RUN_LOCAL   = True             # §2
RUN_EXPORT  = True             # §3
WINDOW_MONTHS = 3              # global metrics and communities on the trailing window ending each month

# ---- hubs and graph rules
HUB_MULT     = 500             # customer hub: undirected on-us degree ≥ HUB_MULT × median degree (as geo)
ENT_HUB_DEG  = 10000           # entity hub: ≥ this many scope customers in the window (or hub_tier 3)
PROJ_CAP     = 100             # projected view: a shared middle node with more scope families is dropped
REACH_MID_MAX = 200            # reach2_int: middle nodes with more on-us partners than this are not used as a hop

# ---- stress and dynamics
STRESS_DROP     = 0.30         # stressed: this month's flow ≤ (1 − STRESS_DROP) × median of the previous 3 months
STRESS_MIN_BASE = 10000        # … and that median ≥ this many dollars (small payers are not judged)
GROW = 0.20                    # growing / shrinking tie: ≥ 20% above / below its 3-month median
STATE_GAP = 3                  # a tie silent this many months ends (tie age restarts)
KEEP_STATES = 2                # pair-state months kept on HDFS
KEEP_EC = 4                    # monthly node-edge caches kept (stress needs 3 months back)

# ---- cluster (no effect on the numbers)
SHUFFLE_PARTITIONS = 2000
RESOURCE_PROFILE = "acc"
RESOURCES = {
    "acc":      {"spark.executor.memory": "4g", "spark.executor.memoryOverhead": "1g", "spark.executor.cores": "1",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "2g"},
    "standard": {"spark.executor.memory": "8g", "spark.executor.memoryOverhead": "3g", "spark.executor.cores": "2",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "4g"},
}
ACC_LIMITS = {"spark.executor.memory": "4g", "spark.executor.memoryOverhead": "1g", "spark.driver.memory": "4g",
              "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g", "spark.dynamicAllocation.maxExecutors": "512"}
HDFS_REPLICATION = 2
NEW_SESSION_EACH_MONTH = True
NET_VERSION = "net-v1.0"

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)
warnings.filterwarnings("ignore")

import os, math, time, json, hashlib, threading, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

_b = SparkSession.builder.appName("pkg_network_metrics").config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
for _k, _v in RESOURCES[RESOURCE_PROFILE].items():
    _b = _b.config(_k, _v)
_b = (_b.config("spark.hadoop.dfs.replication", str(HDFS_REPLICATION))
        .config("spark.dynamicAllocation.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.ui.retainedJobs", "100").config("spark.ui.retainedStages", "100")
        .config("spark.ui.retainedTasks", "10000").config("spark.sql.ui.retainedExecutions", "20")
        .enableHiveSupport())
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")

def start_session():
    s = _b.getOrCreate()
    s.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    s.conf.set("spark.sql.ansi.enabled", "false")
    s.sparkContext.setLogLevel("ERROR")
    try:
        _L = s._jvm.org.apache.logging.log4j
        for _n in _QUIET:
            _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
    except Exception:
        try:
            _L1 = s._jvm.org.apache.log4j
            for _n in _QUIET:
                _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
        except Exception:
            pass
    s._jsc.hadoopConfiguration().set("dfs.replication", str(HDFS_REPLICATION))
    return s

spark = start_session()
HCONF = spark._jsc.hadoopConfiguration()

def new_session():
    """Stop the Spark session and start a fresh one (a new YARN application): nothing piles up in the driver."""
    global spark, HCONF
    try:
        spark.catalog.clearCache()
    except Exception:
        pass
    spark.stop()
    import gc; gc.collect()
    spark = start_session()
    HCONF = spark._jsc.hadoopConfiguration()

pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.1f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body=""):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text, color=None):
    st = f"; color:{color}" if color else ""
    return f"<p style='margin:4px 0; font-size:13px{st}'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS through the JVM (HCONF is looked up at call time, so a new session is picked up)
JPath = spark._jvm.org.apache.hadoop.fs.Path
def fs_for(p):
    return JPath(p).getFileSystem(HCONF)
def exists(p):
    return fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def rename(a, b):
    return fs_for(a).rename(JPath(a), JPath(b))
def ls_dirs(p):
    if not exists(p):
        return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if s.isDirectory()]
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def write_json(p, obj):
    s = fs_for(p).create(JPath(p), True)
    s.write(bytearray(json.dumps(obj, default=str).encode())); s.close()
def read_json(p):
    if not exists(p):
        return None
    s = fs_for(p).open(JPath(p))
    try:
        return json.loads(spark._jvm.org.apache.commons.io.IOUtils.toString(s, "UTF-8"))
    finally:
        s.close()
def table_exists(t):
    db, name = t.split(".")
    return spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").count() > 0
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")
def md5(o):
    return hashlib.md5(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()[:12]
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def month_range(a, b):
    r, m = [], a
    while m <= b:
        r.append(m); m = month_add(m, 1)
    return r


DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
LOC = {k: f"{DB_LOC}/{t.split('.')[1]}" for k, t in OUT_TABLES.items()}
WORK = f"{DB_LOC}/_tmp_pkg_network"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(NET_DIR, exist_ok=True)

# ---- what the session really got
def _gb(v):
    v = str(v).strip().lower()
    return float(v[:-1]) / (1024 if v.endswith("m") else 1) if v[-1:] in ("g", "m") else float(v)
_conf = dict(spark.sparkContext.getConf().getAll())
_rr = []
for _k in sorted(set(RESOURCES[RESOURCE_PROFILE]) | set(ACC_LIMITS)):
    _use = _conf.get(_k, "(default)")
    _lim = ACC_LIMITS.get(_k)
    _over = (_lim is not None and _use != "(default)"
             and (_gb(_use) > _gb(_lim) if _k != "spark.dynamicAllocation.maxExecutors" else int(_use) > int(_lim)))
    _rr.append({"setting": _k, "profile": RESOURCES[RESOURCE_PROFILE].get(_k, ""), "in use": _use,
                "ACC limit": _lim or "", "over ACC": "YES" if _over else ""})
_rr = pd.DataFrame(_rr)
_miss = _rr[(_rr.profile != "") & (_rr["in use"] != _rr.profile)]
out(sec(f"Spark {spark.version} · profile {RESOURCE_PROFILE} · pandas {pd.__version__} · numpy {np.__version__} · "
        f"shared folder {NET_DIR}"),
    para(f"The session was created before this cell: {len(_miss)} setting(s) differ from the profile. Restart the "
         "kernel and run this notebook from the top.", "#a00") if len(_miss) else "",
    tbl(_rr))

# ---- local files on the driver host (the shared folder)
def copy_to_local(hdfs_dir, local_dir):
    """HDFS folder → local folder on this host (the GPU job reads it)."""
    import shutil
    shutil.rmtree(local_dir, ignore_errors=True)
    os.makedirs(os.path.dirname(local_dir), exist_ok=True)
    fs_for(hdfs_dir).copyToLocalFile(False, JPath(hdfs_dir), JPath("file://" + local_dir), True)
    for root, _, files in os.walk(local_dir):            # drop Hadoop checksum files
        for f_ in files:
            if f_.endswith(".crc"):
                os.remove(os.path.join(root, f_))

## §1 Inputs and plan

Columns and id types, published edge months, customers in scope and their families, and which months / windows are
already done with the current settings.

In [ ]:
# §1  Inputs and plan
def need(table, cols):
    dt_ = dict(spark.table(table).dtypes)
    miss = [c for c in cols if c not in dt_]
    assert not miss, f"{table}: missing {miss}; has {sorted(dt_)}"
    return dt_
def partitions(table, key):
    vals = set()
    try:
        for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
            for kv in r[0].split("/"):
                if kv.startswith(key + "="):
                    vals.add(kv.split("=", 1)[1])
    except Exception:
        vals = {str(r[0]) for r in spark.table(table).select(key).distinct().collect()}
    return sorted(v for v in vals if v and v != "None")

edt = need(T_EDGE_M, ["src_id", "src_kind", "dst_id", "dst_kind", "is_self", "amount", "month"])
cdt = need(T_CUST, ["mdm_id", "party_type", "is_tm", "is_pnc_own", "master_pwr_id"])
mdt = need(T_MAP, ["cpty_id", "ent_l2"])
ndt = need(T_ENT, ["ent_id", "name_class"])
pdt = need(T_CPM, ["ent_id", "hub_tier", "month"])
bad = {f"{t}.{c}": d[c] for t, d, cols in [(T_CUST, cdt, ["mdm_id"]), (T_EDGE_M, edt, ["src_id", "dst_id"]),
                                           (T_MAP, mdt, ["cpty_id", "ent_l2"])] for c in cols if d[c] != "string"}
assert not bad, f"id columns must be string (2026-07 incident): {bad}"

E_LOC = table_location(T_EDGE_M)
ALL_MONTHS = [m for m in partitions(T_EDGE_M, "month") if exists(f"{E_LOC}/month={m}/_PKG_BUILD")]
LAST = LAST_MONTH or ALL_MONTHS[-1]
ALL_MONTHS = [m for m in ALL_MONTHS if m <= LAST]
RUN_MONTHS = ALL_MONTHS[-MONTHS_BACK:] if MONTHS_BACK else ALL_MONTHS
CPM_MONTHS = partitions(T_CPM, "month")

SIG = md5({"v": NET_VERSION, "hub": HUB_MULT, "ehub": ENT_HUB_DEG, "cap": PROJ_CAP, "reach": REACH_MID_MAX,
           "sd": STRESS_DROP, "smin": STRESS_MIN_BASE, "grow": GROW, "gap": STATE_GAP, "win": WINDOW_MONTHS})
def local_done(m):
    mk = read_json(f"{LOC['local']}/month={m}/_NET_BUILD")
    return bool(mk) and mk.get("sig") == SIG and exists(f"{LOC['local']}/month={m}/_SUCCESS")
def export_done(m):
    p = os.path.join(NET_DIR, f"w={m}", "_EXPORTED")
    if not os.path.exists(p):
        return False
    try:
        return json.load(open(p)).get("sig") == SIG
    except Exception:
        return False
TODO = [m for m in RUN_MONTHS if (RUN_LOCAL and not local_done(m)) or (RUN_EXPORT and not export_done(m))]
_fin = os.path.join(NET_DIR, "_EXPORT_FINISHED")
if TODO and os.path.exists(_fin):
    os.remove(_fin)                     # the GPU job waits for new windows until this marker is back

# customers: scope, family
CA = (spark.table(T_CUST)
        .select(F.concat(F.lit("C:"), F.trim(F.col("mdm_id"))).alias("key"),
                F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"),
                (F.col("party_type") == "O").alias("is_org"),
                F.coalesce(F.col("is_pnc_own").cast("boolean"), F.lit(False)).alias("is_pnc_own"),
                F.when(F.length(F.trim(F.col("master_pwr_id").cast("string"))) > 0,
                       F.concat(F.lit("F:"), F.trim(F.col("master_pwr_id").cast("string"))))
                 .otherwise(F.concat(F.lit("M:"), F.trim(F.col("mdm_id")))).alias("family_id"))
        .withColumn("scope", F.col("is_tm") | F.coalesce(F.col("is_org"), F.lit(False)))
        .select("key", "scope", "is_pnc_own", "family_id"))
_dup = CA.groupBy("key").count().where("count > 1").count()
assert _dup == 0, f"{T_CUST}: {_dup} mdm_id with more than one row"
CA.write.mode("overwrite").parquet(f"{WORK}/ca")
MP = (spark.table(T_MAP).select(F.trim(F.col("cpty_id")).alias("cpty"), F.col("ent_l2").cast("string").alias("ent"))
        .groupBy("cpty").agg(F.min("ent").alias("ent")))
MP.write.mode("overwrite").parquet(f"{WORK}/mp")
EA = spark.table(T_ENT).select(F.concat(F.lit("E:"), F.col("ent_id").cast("string")).alias("key"), "name_class")
EA.write.mode("overwrite").parquet(f"{WORK}/ea")
ca = spark.read.parquet(f"{WORK}/ca")
fam = ca.where("scope").groupBy("family_id").count()
fs_ = fam.agg(F.count("*").alias("families"), F.sum((F.col("count") > 1).cast("long")).alias("multi"),
              F.sum(F.when(F.col("count") > 1, F.col("count")).otherwise(0)).alias("in_multi")).first()
sc_n = ca.where("scope").count()
out(sec(f"§1 Inputs · {len(ALL_MONTHS)} published months ({ALL_MONTHS[0]} → {ALL_MONTHS[-1]}) · signature {SIG}"),
    tbl(pd.DataFrame([
        {"item": "scope customers (TM or organisation)", "value": f"{sc_n:,}"},
        {"item": "their families (master power id; none = own family)", "value": f"{fs_['families']:,}, of which {fs_['multi']:,} with 2+ members ({fs_['in_multi']:,} customers)"},
        {"item": "months in this run", "value": f"{len(RUN_MONTHS)} ({RUN_MONTHS[0]} → {RUN_MONTHS[-1]})"},
        {"item": "months to build now", "value": f"{len(TODO)}" + (f" ({TODO[0]} → {TODO[-1]})" if TODO else "")},
        {"item": "shared folder for the GPU job", "value": NET_DIR}])))

## §2–§3 Month loop: local metrics, then the window export

**Per month m.** Node edges of m (customers `C:mdm_id`, outside entities `E:ent_l2`; categories merged; own accounts
out) are cached with their node totals (last `KEEP_EC` months). Local metrics per scope customer:

| column | definition |
|---|---|
| `reach2_int` | scope customers within 2 on-us hops (no hubs, no family edges, middles with ≤ `REACH_MID_MAX` partners) |
| `shared_cpty_share_{in,out}`, `shared_cpty_amt_share_{in,out}` | share of outside entities (and their dollars) another scope customer also uses this month |
| `recip_share`, `recip_amt_share` | share of on-us partners (and dollars) with payments both ways |
| `nbr_size_{in,out}` | dollar-weighted mean log10(1 + counterparty's total PNC flow this month) |
| `stressed_in_share`, `stressed_out_share`, `n_stressed_payers` | share of the customer's previous-3-month inflow (outflow) from payers (payees) whose PNC-wide outflow (inflow) this month is ≤ (1 − `STRESS_DROP`) × their 3-month median (median ≥ `STRESS_MIN_BASE`) |
| `exposure2_in` | Σ over payer customers of their share of the customer's previous-3-month inflow × their own `stressed_in_share` |
| `tie_age_mean`, `amt_share_ties_12m` | dollar-weighted months since the tie started (a gap of `STATE_GAP` months restarts it; measured from 2024-01), share on ties ≥ 12 months |
| `turnover_1m`, `turnover_3m` | 1 − dollar-weighted Jaccard of the counterparty set (both directions) against 1 and 3 months earlier |
| `amt_share_growing`, `amt_share_shrinking` | share of the 3-month baseline on ties ≥ `GROW` above / below their 3-month median this month |
| `intra_family_amt_share` | share of the customer's dollars with its own family (other customers with the same master power id) |

**Window export (3 months ending m).** Hubs, then three graphs with integer ids (`xxhash64` of the key; a collision
stops the run): `int_edges` (scope customers, directed), `obs_edges` (scope customers + non-person entities with ≥ 2
scope customers, directed), `com_edges` (families, undirected: projected through shared middles with ≤ `PROJ_CAP`
scope families, weight Σ min(months) / (d − 1) and Σ log(1 + min dollars) / (d − 1), plus direct on-us family pairs),
`nodes`, `fams`, `hubs`. Copied to `NET_DIR/w=m/`, then the `_EXPORTED` marker.

In [ ]:
# §2–§3  Month loop
LOG = []
_PH = display(HTML(""), display_id=True)
_CUR = {"m": None, "step": "", "t": None}
_PLOCK = threading.Lock()
def _panel(final=None):
    with _PLOCK:
        lg = pd.DataFrame(LOG)
        n_done = len(lg)
        mpm = lg["minutes"].mean() if n_done else None
        left = (len(TODO) - n_done) * mpm if mpm else None
        head = (f"{n_done} of {len(TODO)} month(s) done this run"
                + (f" · {mpm:.1f} min per month · about {left / 60:.1f} h left" if left is not None else ""))
        now = (f"<br>now: <b>{_CUR['m']}</b> · {_CUR['step']} · {(time.time() - _CUR['t']) / 60:.1f} min into the month"
               if _CUR["m"] and final is None else "")
        _PH.update(HTML("<div>" + (final or "") + f"<p style='margin:4px 0'>{head}{now}<br><span style='font-size:12px'>"
                        f"updated {dt.datetime.now().strftime('%H:%M:%S')}</span></p>" + (tbl(lg) if n_done else "") + "</div>"))
_HB = threading.Event()
def _heartbeat():
    while not _HB.wait(30):
        try:
            if _CUR["m"]:
                _panel()
        except Exception:
            pass
_HBT = threading.Thread(target=_heartbeat, daemon=True); _HBT.start()
def _stop_heartbeat():
    _CUR["m"] = None; _HB.set(); _HBT.join(timeout=60)
def step(s_):
    _CUR["step"] = s_; _panel()

# ---------------------------------------------------------------- caches
def ec_path(m):  return f"{WORK}/ec/m={m}"
def tot_path(m): return f"{WORK}/tot/m={m}"
def st_path(m):  return f"{WORK}/state/m={m}"

def build_ec(m):
    """Node edges of month m: a pays b (keys C:/E:), amount; and node totals (out, in)."""
    e = (spark.table(T_EDGE_M).where((F.col("month") == m) & ~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
           .select(F.trim(F.col("src_id")).alias("src"), "src_kind", F.trim(F.col("dst_id")).alias("dst"), "dst_kind",
                   F.col("amount").cast("double").alias("amt")))
    e = e.withColumn("cpty", F.when(F.col("src_kind") == "CPTY", F.col("src")).when(F.col("dst_kind") == "CPTY", F.col("dst")))
    e = e.join(spark.read.parquet(f"{WORK}/mp"), "cpty", "left")
    ekey = F.concat(F.lit("E:"), F.coalesce(F.col("ent"), F.concat(F.lit("U:"), F.col("cpty"))))
    a = F.when(F.col("src_kind") == "CUST", F.concat(F.lit("C:"), F.col("src"))).otherwise(ekey)
    b = F.when(F.col("dst_kind") == "CUST", F.concat(F.lit("C:"), F.col("dst"))).otherwise(ekey)
    ec = e.select(a.alias("a"), b.alias("b"), "amt").where(F.col("a") != F.col("b")).groupBy("a", "b").agg(F.sum("amt").alias("amt"))
    ec.write.mode("overwrite").parquet(ec_path(m))
    ec = spark.read.parquet(ec_path(m))
    o_ = ec.groupBy(F.col("a").alias("node")).agg(F.sum("amt").alias("out_amt"))
    i_ = ec.groupBy(F.col("b").alias("node")).agg(F.sum("amt").alias("in_amt"))
    o_.join(i_, "node", "full").na.fill(0.0, ["out_amt", "in_amt"]).write.mode("overwrite").parquet(tot_path(m))

def ensure_ec(months):
    for mm in months:
        if not (exists(ec_path(mm) + "/_SUCCESS") and exists(tot_path(mm) + "/_SUCCESS")):
            build_ec(mm)

def prune(base, keep):
    for d_ in ls_dirs(base):
        if d_.startswith("m=") and d_[2:] not in keep:
            delete(f"{base}/{d_}")

# ---------------------------------------------------------------- local metrics + state
def side_rows(m):
    """Scope customers' rows of month m: c, dir, o, amt, intra (same family)."""
    ec = spark.read.parquet(ec_path(m))
    ca = spark.read.parquet(f"{WORK}/ca")
    sc = ca.where("scope").select(F.col("key").alias("c"), F.col("family_id").alias("cfam"))
    of = ca.select(F.col("key").alias("o"), F.col("family_id").alias("ofam"))
    o_ = ec.join(sc.withColumnRenamed("c", "a"), "a").select(F.col("a").alias("c"), F.lit("out").alias("dir"), F.col("b").alias("o"), "amt", "cfam")
    i_ = ec.join(sc.withColumnRenamed("c", "b"), "b").select(F.col("b").alias("c"), F.lit("in").alias("dir"), F.col("a").alias("o"), "amt", "cfam")
    s_ = (o_.unionByName(i_).join(of, "o", "left")
            .withColumn("intra", F.coalesce(F.col("ofam") == F.col("cfam"), F.lit(False)))
            .drop("ofam"))
    return s_

def update_state(m, X, prev):
    """State after month m: (c, dir, o, first_month, last_month, a1 = m, a2 = m−1, a3 = m−2), pairs active in the last STATE_GAP months."""
    cur = X.select("c", "dir", "o", F.col("x"))
    if prev is None:
        st = cur.select("c", "dir", "o", F.lit(m).alias("first_month"), F.lit(m).alias("last_month"),
                        F.col("x").alias("a1"), F.lit(0.0).alias("a2"), F.lit(0.0).alias("a3"))
    else:
        p = prev.select("c", "dir", "o", "first_month", "last_month", F.col("a1").alias("p1"), F.col("a2").alias("p2"))
        st = (p.join(cur, ["c", "dir", "o"], "full")
               .select("c", "dir", "o",
                       F.coalesce(F.col("first_month"), F.lit(m)).alias("first_month"),
                       F.when(F.col("x").isNotNull(), F.lit(m)).otherwise(F.col("last_month")).alias("last_month"),
                       F.coalesce(F.col("x"), F.lit(0.0)).alias("a1"), F.coalesce(F.col("p1"), F.lit(0.0)).alias("a2"),
                       F.coalesce(F.col("p2"), F.lit(0.0)).alias("a3")))
    st = st.where(F.col("last_month") > month_add(m, -STATE_GAP))
    st.write.mode("overwrite").parquet(st_path(m))

def months_between(a, b):
    """b − a in months for 'YYYY-MM' columns / literals."""
    return ((F.substring(b, 1, 4).cast("int") * 12 + F.substring(b, 6, 2).cast("int"))
            - (F.substring(a, 1, 4).cast("int") * 12 + F.substring(a, 6, 2).cast("int")))

def build_local(m, write=True):
    prev_m = month_add(m, -1)
    has_prev = prev_m in ALL_MONTHS and exists(st_path(prev_m) + "/_SUCCESS")
    prev = spark.read.parquet(st_path(prev_m)) if has_prev else None
    S = side_rows(m)
    S.write.mode("overwrite").parquet(f"{WORK}/s"); S = spark.read.parquet(f"{WORK}/s")
    Sx = S.where(~F.col("intra"))
    X = Sx.groupBy("c", "dir", "o").agg(F.sum("amt").alias("x"))
    X.write.mode("overwrite").parquet(f"{WORK}/x"); X = spark.read.parquet(f"{WORK}/x")
    if not write:                      # warm-up month: state only
        update_state(m, X, prev)
        return None
    step("local: family, shared, reciprocity, size")
    base = S.groupBy("c").agg(F.first("cfam").alias("family_id"), F.sum("amt").alias("_tot"),
                              F.sum(F.when(F.col("intra"), F.col("amt")).otherwise(0.0)).alias("_intra"))
    base = base.withColumn("intra_family_amt_share", F.when(F.col("_tot") > 0, F.col("_intra") / F.col("_tot"))).drop("_tot", "_intra")
    isE = F.col("o").startswith("E:"); isC = F.col("o").startswith("C:")
    sc_keys = spark.read.parquet(f"{WORK}/ca").where("scope").select(F.col("key").alias("c"))
    ent_deg = X.where(isE).groupBy("o").agg(F.countDistinct("c").alias("edeg"))
    sh = (X.where(isE).join(ent_deg, "o").groupBy("c").pivot("dir", ["in", "out"])
            .agg(F.count("*").alias("n"), F.sum((F.col("edeg") >= 2).cast("long")).alias("ns"),
                 F.sum("x").alias("a"), F.sum(F.when(F.col("edeg") >= 2, F.col("x")).otherwise(0.0)).alias("as")))
    sh = sh.select("c", *[F.when(F.col(f"{d}_n") > 0, F.col(f"{d}_ns") / F.col(f"{d}_n")).alias(f"shared_cpty_share_{d}") for d in ("in", "out")],
                   *[F.when(F.col(f"{d}_a") > 0, F.col(f"{d}_as") / F.col(f"{d}_a")).alias(f"shared_cpty_amt_share_{d}") for d in ("in", "out")])
    rp = (X.where(isC).groupBy("c", "o").agg(F.countDistinct("dir").alias("nd"), F.sum("x").alias("a"))
            .groupBy("c").agg(F.count("*").alias("np"), F.sum((F.col("nd") == 2).cast("long")).alias("nr"),
                              F.sum("a").alias("a"), F.sum(F.when(F.col("nd") == 2, F.col("a")).otherwise(0.0)).alias("ar"))
            .select("c", (F.col("nr") / F.col("np")).alias("recip_share"),
                    F.when(F.col("a") > 0, F.col("ar") / F.col("a")).alias("recip_amt_share")))
    tot = spark.read.parquet(tot_path(m))
    nb = (X.join(tot.select(F.col("node").alias("o"), F.log10(1 + F.col("out_amt") + F.col("in_amt")).alias("lsz")), "o", "left")
            .groupBy("c").pivot("dir", ["in", "out"]).agg((F.sum(F.col("x") * F.col("lsz")) / F.sum("x")).alias("v")))
    nb = nb.select("c", F.col("in").alias("nbr_size_in"), F.col("out").alias("nbr_size_out"))

    step("local: stress and exposure")
    prior3 = [month_add(m, -k) for k in (1, 2, 3)]
    if prev is not None and all(p_ in ALL_MONTHS for p_ in prior3):
        ensure_ec(prior3)
        t0_ = tot.select(F.col("node").alias("o"), F.col("out_amt").alias("o0"), F.col("in_amt").alias("i0"))
        for k, p_ in enumerate(prior3, 1):
            t0_ = t0_.join(spark.read.parquet(tot_path(p_)).select(F.col("node").alias("o"), F.col("out_amt").alias(f"o{k}"),
                                                                   F.col("in_amt").alias(f"i{k}")), "o", "full")
        t0_ = t0_.na.fill(0.0)
        med = lambda p: F.sort_array(F.array(F.col(f"{p}1"), F.col(f"{p}2"), F.col(f"{p}3")))[1]
        strs = t0_.select("o", ((med("o") >= STRESS_MIN_BASE) & (F.col("o0") <= (1 - STRESS_DROP) * med("o"))).alias("sp"),
                          ((med("i") >= STRESS_MIN_BASE) & (F.col("i0") <= (1 - STRESS_DROP) * med("i"))).alias("se"))
        W = (prev.select("c", "dir", "o", (F.col("a1") + F.col("a2") + F.col("a3")).alias("w"))
                 .where(F.col("w") > 0).join(strs, "o", "left").na.fill(False, ["sp", "se"]))
        W = W.join(spark.read.parquet(f"{WORK}/ca").select(F.col("key").alias("o"), F.col("family_id").alias("ofam")), "o", "left")
        W = (W.join(base.select("c", F.col("family_id").alias("cfam")), "c")
              .where(~F.coalesce(F.col("ofam") == F.col("cfam"), F.lit(False))))
        W.write.mode("overwrite").parquet(f"{WORK}/w"); W = spark.read.parquet(f"{WORK}/w")
        stv = (W.groupBy("c").agg(
                    F.sum(F.when(F.col("dir") == "in", F.col("w")).otherwise(0.0)).alias("win"),
                    F.sum(F.when((F.col("dir") == "in") & F.col("sp"), F.col("w")).otherwise(0.0)).alias("win_s"),
                    F.sum(F.when(F.col("dir") == "out", F.col("w")).otherwise(0.0)).alias("wout"),
                    F.sum(F.when((F.col("dir") == "out") & F.col("se"), F.col("w")).otherwise(0.0)).alias("wout_s"),
                    F.sum(((F.col("dir") == "in") & F.col("sp")).cast("long")).alias("n_stressed_payers"))
                 .select("c", F.when(F.col("win") > 0, F.col("win_s") / F.col("win")).alias("stressed_in_share"),
                         F.when(F.col("wout") > 0, F.col("wout_s") / F.col("wout")).alias("stressed_out_share"),
                         "n_stressed_payers", "win"))
        stv.write.mode("overwrite").parquet(f"{WORK}/stv"); stv = spark.read.parquet(f"{WORK}/stv")
        ex2 = (W.where((F.col("dir") == "in") & F.col("o").startswith("C:"))
                .join(stv.select(F.col("c").alias("o"), F.col("stressed_in_share").alias("ps")), "o", "left")
                .groupBy("c").agg(F.sum(F.col("w") * F.coalesce(F.col("ps"), F.lit(0.0))).alias("e2"))
                .join(stv.select("c", "win"), "c", "right")
                .select("c", F.when(F.col("win") > 0, F.coalesce(F.col("e2"), F.lit(0.0)) / F.col("win")).alias("exposure2_in")))
        stv = stv.drop("win").join(ex2, "c", "left")
    else:
        stv = sc_keys.limit(0).select("c", *[F.lit(None).cast(t_).alias(n_) for n_, t_ in
              (("stressed_in_share", "double"), ("stressed_out_share", "double"), ("n_stressed_payers", "long"), ("exposure2_in", "double"))])

    step("local: dynamics")
    if prev is not None:
        P = prev.select("c", "dir", "o", "first_month", F.col("a1").alias("p1"), F.col("a2").alias("p2"), F.col("a3").alias("p3"))
        J = (P.join(X, ["c", "dir", "o"], "full")
              .withColumn("x", F.coalesce("x", F.lit(0.0)))
              .na.fill(0.0, ["p1", "p2", "p3"])
              .withColumn("age", F.when(F.col("x") > 0, months_between(F.coalesce(F.col("first_month"), F.lit(m)), F.lit(m))))
              .withColumn("med", F.sort_array(F.array("p1", "p2", "p3"))[1]))
        dyn = (J.groupBy("c").agg(
                  F.sum(F.when(F.col("x") > 0, F.col("x") * F.col("age"))).alias("xa"), F.sum("x").alias("sx"),
                  F.sum(F.when(F.col("age") >= 12, F.col("x")).otherwise(0.0)).alias("x12"),
                  F.sum(F.least("x", "p1")).alias("mn1"), F.sum(F.greatest("x", "p1")).alias("mx1"),
                  F.sum(F.least("x", "p3")).alias("mn3"), F.sum(F.greatest("x", "p3")).alias("mx3"),
                  F.sum("med").alias("smed"),
                  F.sum(F.when((F.col("med") > 0) & (F.col("x") >= (1 + GROW) * F.col("med")), F.col("med")).otherwise(0.0)).alias("g"),
                  F.sum(F.when((F.col("med") > 0) & (F.col("x") <= (1 - GROW) * F.col("med")), F.col("med")).otherwise(0.0)).alias("sh"))
               .select("c", F.when(F.col("sx") > 0, F.col("xa") / F.col("sx")).alias("tie_age_mean"),
                       F.when(F.col("sx") > 0, F.col("x12") / F.col("sx")).alias("amt_share_ties_12m"),
                       F.when(F.col("mx1") > 0, 1 - F.col("mn1") / F.col("mx1")).alias("turnover_1m"),
                       F.when(F.col("mx3") > 0, 1 - F.col("mn3") / F.col("mx3")).alias("turnover_3m"),
                       F.when(F.col("smed") > 0, F.col("g") / F.col("smed")).alias("amt_share_growing"),
                       F.when(F.col("smed") > 0, F.col("sh") / F.col("smed")).alias("amt_share_shrinking")))
        if month_add(m, -3) not in ALL_MONTHS:
            dyn = dyn.withColumn("turnover_3m", F.lit(None).cast("double"))
    else:
        dyn = (X.groupBy("c").agg(F.lit(0.0).alias("tie_age_mean"))
                .select("c", "tie_age_mean", *[F.lit(None).cast("double").alias(n_) for n_ in
                        ("amt_share_ties_12m", "turnover_1m", "turnover_3m", "amt_share_growing", "amt_share_shrinking")]))

    step("local: 2-hop reach")
    ec = spark.read.parquet(ec_path(m))
    ca = spark.read.parquet(f"{WORK}/ca")
    U = ec.where(F.col("a").startswith("C:") & F.col("b").startswith("C:")).select(
        F.least("a", "b").alias("u"), F.greatest("a", "b").alias("v")).distinct()
    U = (U.join(ca.select(F.col("key").alias("u"), F.col("family_id").alias("fu"), F.col("is_pnc_own").alias("pu")), "u")
          .join(ca.select(F.col("key").alias("v"), F.col("family_id").alias("fv"), F.col("is_pnc_own").alias("pv")), "v")
          .where((F.col("fu") != F.col("fv")) & ~F.col("pu") & ~F.col("pv")).select("u", "v"))
    D = U.select(F.col("u").alias("n")).unionAll(U.select(F.col("v").alias("n"))).groupBy("n").count()
    med_deg = D.approxQuantile("count", [0.5], 0.001)[0] if D.take(1) else 1.0
    hubs_m = D.where(F.col("count") >= HUB_MULT * max(med_deg, 1.0)).select("n")
    A = (U.select(F.col("u").alias("x"), F.col("v").alias("y")).unionAll(U.select(F.col("v").alias("x"), F.col("u").alias("y")))
          .join(hubs_m.withColumnRenamed("n", "x"), "x", "left_anti").join(hubs_m.withColumnRenamed("n", "y"), "y", "left_anti"))
    A.write.mode("overwrite").parquet(f"{WORK}/adj"); A = spark.read.parquet(f"{WORK}/adj")
    small = D.where(F.col("count") <= REACH_MID_MAX).select(F.col("n").alias("mid"))
    one = A.join(sc_keys.withColumnRenamed("c", "x"), "x").select(F.col("x").alias("c"), F.col("y").alias("r"))
    two = (A.join(sc_keys.withColumnRenamed("c", "x"), "x").select(F.col("x").alias("c"), F.col("y").alias("mid"))
             .join(small, "mid").join(A.select(F.col("x").alias("mid"), F.col("y").alias("r")), "mid").select("c", "r"))
    rch = (one.unionAll(two).where(F.col("c") != F.col("r"))
              .join(sc_keys.withColumnRenamed("c", "r"), "r").groupBy("c").agg(F.countDistinct("r").alias("reach2_int")))

    step("local: write")
    R = (base.join(rch, "c", "left").join(sh, "c", "left").join(rp, "c", "left").join(nb, "c", "left")
             .join(stv, "c", "left").join(dyn, "c", "left")
             .withColumn("reach2_int", F.coalesce(F.col("reach2_int"), F.lit(0)).cast("long"))
             .withColumn("mdm_id", F.substring(F.col("c"), 3, 1000)).drop("c"))
    cols = ["mdm_id", "family_id", "reach2_int", "shared_cpty_share_in", "shared_cpty_share_out", "shared_cpty_amt_share_in",
            "shared_cpty_amt_share_out", "recip_share", "recip_amt_share", "nbr_size_in", "nbr_size_out", "stressed_in_share",
            "stressed_out_share", "n_stressed_payers", "exposure2_in", "tie_age_mean", "amt_share_ties_12m", "turnover_1m",
            "turnover_3m", "amt_share_growing", "amt_share_shrinking", "intra_family_amt_share"]
    p = f"{LOC['local']}/month={m}"
    delete(p)
    R.select(*cols).write.mode("overwrite").parquet(p)
    update_state(m, X, prev)
    n = spark.read.parquet(p).count()
    # reconciliation: customer-side dollars = edge dollars with a scope customer on that side
    s_tot = spark.read.parquet(f"{WORK}/s").groupBy("dir").agg(F.sum("amt").alias("a")).toPandas().set_index("dir")["a"].to_dict()
    e_tot = (ec.join(sc_keys.withColumnRenamed("c", "a"), "a").agg(F.sum("amt")).first()[0] or 0.0,
             ec.join(sc_keys.withColumnRenamed("c", "b"), "b").agg(F.sum("amt")).first()[0] or 0.0)
    assert abs(s_tot.get("out", 0.0) - e_tot[0]) <= 1e-6 * max(1.0, e_tot[0]) and abs(s_tot.get("in", 0.0) - e_tot[1]) <= 1e-6 * max(1.0, e_tot[1]), \
        f"{m}: side rows {s_tot} ≠ edges out {e_tot[0]} / in {e_tot[1]}"
    write_json(f"{p}/_NET_BUILD", {"sig": SIG, "built_at": dt.datetime.now().isoformat(timespec="seconds"), "rows": n,
                                   "stress": prev is not None})
    return n

# ---------------------------------------------------------------- window export
def hid(c):
    return F.xxhash64(c)

def export_window(m):
    W_ = [mm for mm in ALL_MONTHS if month_add(m, -WINDOW_MONTHS) < mm <= m]
    ensure_ec(W_)
    ca = spark.read.parquet(f"{WORK}/ca")
    ea = spark.read.parquet(f"{WORK}/ea")
    ew = None
    for mm in W_:
        x_ = spark.read.parquet(ec_path(mm)).withColumn("month", F.lit(mm))
        ew = x_ if ew is None else ew.unionByName(x_)
    EW = ew.groupBy("a", "b").agg(F.countDistinct("month").cast("int").alias("wm"), F.sum("amt").alias("usd"))
    EW.write.mode("overwrite").parquet(f"{WORK}/ew"); EW = spark.read.parquet(f"{WORK}/ew")
    sc = ca.where("scope").select(F.col("key").alias("n"), "family_id", "is_pnc_own")
    # customer hubs: undirected on-us degree in the window
    U = EW.where(F.col("a").startswith("C:") & F.col("b").startswith("C:")).select(F.least("a", "b").alias("u"), F.greatest("a", "b").alias("v")).distinct()
    D = U.select(F.col("u").alias("n")).unionAll(U.select(F.col("v").alias("n"))).groupBy("n").agg(F.count("*").alias("deg"))
    med = D.approxQuantile("deg", [0.5], 0.001)[0] if D.take(1) else 1.0
    cut = HUB_MULT * max(med, 1.0)
    ch = (D.where(F.col("deg") >= cut).select("n", "deg", F.lit("degree").alias("reason"))
           .unionByName(ca.where("is_pnc_own").select(F.col("key").alias("n")).join(D, "n", "left")
                        .select("n", F.coalesce("deg", F.lit(0)).alias("deg"), F.lit("pnc_own").alias("reason")))
           .groupBy("n").agg(F.max("deg").alias("deg"), F.min("reason").alias("reason")))
    # entities: scope degree, tier, class
    es = (EW.where(F.col("a").startswith("E:")).select(F.col("a").alias("e"), F.col("b").alias("n"))
            .unionByName(EW.where(F.col("b").startswith("E:")).select(F.col("b").alias("e"), F.col("a").alias("n")))
            .join(sc.select("n"), "n").groupBy("e").agg(F.countDistinct("n").alias("sdeg")))
    hm = max([mm for mm in CPM_MONTHS if mm <= m], default=None)
    tier = (spark.table(T_CPM).where(F.col("month") == hm).select(F.concat(F.lit("E:"), F.col("ent_id").cast("string")).alias("e"), "hub_tier")
              .groupBy("e").agg(F.max("hub_tier").alias("hub_tier")) if hm else None)
    es = es.join(ea.withColumnRenamed("key", "e"), "e", "left")
    if tier is not None:
        es = es.join(tier, "e", "left")
    else:
        es = es.withColumn("hub_tier", F.lit(None).cast("int"))
    es = es.withColumn("is_hub", (F.coalesce(F.col("hub_tier"), F.lit(0)) >= 3) | (F.col("sdeg") >= ENT_HUB_DEG))
    es.write.mode("overwrite").parquet(f"{WORK}/es"); es = spark.read.parquet(f"{WORK}/es")
    eh = es.where("is_hub").select(F.col("e").alias("n"), F.col("sdeg").alias("deg"),
                                   F.when(F.coalesce(F.col("hub_tier"), F.lit(0)) >= 3, "tier3").otherwise("degree").alias("reason"))
    HUBS = ch.withColumn("kind", F.lit("C")).unionByName(eh.withColumn("kind", F.lit("E")))
    HUBS.write.mode("overwrite").parquet(f"{WORK}/hubs"); HUBS = spark.read.parquet(f"{WORK}/hubs")
    SC = sc.join(ch.select("n"), "n", "left_anti")                                         # scope customers, not hubs
    EN = es.where(~F.col("is_hub") & (F.col("sdeg") >= 2) & (F.coalesce(F.col("name_class"), F.lit("")) != "person")).select(F.col("e").alias("n"))
    NODES = (SC.select("n", F.lit("C").alias("kind"), "family_id")
               .unionByName(EN.select("n", F.lit("E").alias("kind"), F.lit(None).cast("string").alias("family_id"))))
    tw = (EW.groupBy(F.col("a").alias("n")).agg(F.sum("usd").alias("out_usd"))
            .join(EW.groupBy(F.col("b").alias("n")).agg(F.sum("usd").alias("in_usd")), "n", "full"))
    NODES = NODES.join(tw, "n", "left").na.fill(0.0, ["out_usd", "in_usd"]).withColumn("nid", hid(F.col("n")))
    NODES = NODES.withColumn("fid", F.when(F.col("family_id").isNotNull(), hid(F.col("family_id"))))
    NODES.write.mode("overwrite").parquet(f"{WORK}/nodes"); NODES = spark.read.parquet(f"{WORK}/nodes")
    nn = NODES.agg(F.count("*").alias("n"), F.countDistinct("nid").alias("u")).first()
    assert nn["n"] == nn["u"], f"{m}: id collision in node ids ({nn['n'] - nn['u']})"
    nk = NODES.select("n", "kind", "family_id", "nid")
    # internal and observed edges (no family pairs)
    e2 = (EW.join(nk.select(F.col("n").alias("a"), F.col("kind").alias("ka"), F.col("family_id").alias("fa"), F.col("nid").alias("src")), "a")
            .join(nk.select(F.col("n").alias("b"), F.col("kind").alias("kb"), F.col("family_id").alias("fb"), F.col("nid").alias("dst")), "b")
            .where(~F.coalesce(F.col("fa") == F.col("fb"), F.lit(False))))
    e2.write.mode("overwrite").parquet(f"{WORK}/e2"); e2 = spark.read.parquet(f"{WORK}/e2")
    INT = e2.where((F.col("ka") == "C") & (F.col("kb") == "C")).select("src", "dst", "wm", "usd")
    OBS = e2.select("src", "dst", "wm", "usd")
    # community graph on families: projected through shared middles + direct on-us family pairs
    scf = SC.select(F.col("n").alias("c"), F.col("family_id").alias("f"))
    mids_ok = EN.select(F.col("n").alias("x")).unionByName(SC.select(F.col("n").alias("x")))
    mr = (EW.select(F.col("a").alias("c"), F.col("b").alias("x"), "wm", "usd")
            .unionByName(EW.select(F.col("b").alias("c"), F.col("a").alias("x"), "wm", "usd"))
            .join(scf, "c").join(mids_ok, "x")
            .join(ca.select(F.col("key").alias("x"), F.col("family_id").alias("fx")), "x", "left")
            .where(~F.coalesce(F.col("fx") == F.col("f"), F.lit(False)))
            .groupBy("x", "f").agg(F.max("wm").alias("wm"), F.sum("usd").alias("usd")))
    dx = mr.groupBy("x").agg(F.count("*").alias("d")).where((F.col("d") >= 2) & (F.col("d") <= PROJ_CAP))
    mr = mr.join(dx, "x")
    mr.write.mode("overwrite").parquet(f"{WORK}/mr"); mr = spark.read.parquet(f"{WORK}/mr")
    pj = (mr.alias("p").join(mr.alias("q"), "x").where(F.col("p.f") < F.col("q.f"))
            .select(F.col("p.f").alias("f1"), F.col("q.f").alias("f2"),
                    (F.least(F.col("p.wm"), F.col("q.wm")) / (F.col("p.d") - 1)).alias("w_months"),
                    (F.log1p(F.least(F.col("p.usd"), F.col("q.usd"))) / (F.col("p.d") - 1)).alias("w_logusd")))
    dfam = (e2.where((F.col("ka") == "C") & (F.col("kb") == "C"))
              .select(F.least("fa", "fb").alias("f1"), F.greatest("fa", "fb").alias("f2"), "wm", "usd")
              .groupBy("f1", "f2").agg(F.max("wm").cast("double").alias("w_months"), F.log1p(F.sum("usd")).alias("w_logusd")))
    COM = (pj.unionByName(dfam).groupBy("f1", "f2").agg(F.sum("w_months").alias("w_months"), F.sum("w_logusd").alias("w_logusd"))
             .select(hid(F.col("f1")).alias("src"), hid(F.col("f2")).alias("dst"), "w_months", "w_logusd"))
    FAMS = scf.groupBy(F.col("f").alias("family_id")).agg(F.count("*").alias("n_members")).withColumn("fid", hid(F.col("family_id")))
    FAMS.write.mode("overwrite").parquet(f"{WORK}/fams"); FAMS = spark.read.parquet(f"{WORK}/fams")
    ff = FAMS.agg(F.count("*").alias("n"), F.countDistinct("fid").alias("u")).first()
    assert ff["n"] == ff["u"], f"{m}: id collision in family ids ({ff['n'] - ff['u']})"
    # write to HDFS, copy to the shared folder
    base = f"{WORK}/export"
    delete(base)
    parts = {"nodes": (NODES.select("nid", F.col("n").alias("key"), "kind", "family_id", "fid", "out_usd", "in_usd"), 8),
             "fams": (FAMS, 4), "int_edges": (INT, 16), "obs_edges": (OBS, 32), "com_edges": (COM, 32),
             "hubs": (HUBS.select(F.col("n").alias("key"), "kind", "reason", "deg"), 2)}
    cnt = {}
    for k_, (df_, np_) in parts.items():
        df_.repartition(np_).write.mode("overwrite").parquet(f"{base}/{k_}")
        cnt[k_] = spark.read.parquet(f"{base}/{k_}").count()
    loc = os.path.join(NET_DIR, f"w={m}")
    copy_to_local(base, loc)
    json.dump({"sig": SIG, "months": W_, "counts": cnt, "hub_degree_cut": cut,
               "exported_at": dt.datetime.now().isoformat(timespec="seconds")}, open(os.path.join(loc, "_EXPORTED"), "w"))
    row = {"window_end": m, "months": len(W_), **{f"n_{k_}": v_ for k_, v_ in cnt.items()}, "hub_degree_cut": cut}
    pd.DataFrame([row]).to_csv(f"{OUT_DIR}/graph_sizes.csv", mode="a", header=not os.path.exists(f"{OUT_DIR}/graph_sizes.csv"), index=False)
    return cnt

def register_local():
    k, t = "local", OUT_TABLES["local"]
    parts_ = sorted(x for x in ls_dirs(LOC[k]) if x.startswith("month="))
    if not parts_:
        return
    sch = spark.read.parquet(f"{LOC[k]}/{parts_[-1]}").schema
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
    if table_exists(t) and [c for c in spark.table(t).columns if c != "month"] != [f.name for f in sch.fields]:
        spark.sql(f"DROP TABLE {t}")
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {t} ({cols}) PARTITIONED BY (month string) STORED AS PARQUET LOCATION '{LOC[k]}'")
    spark.sql(f"ALTER TABLE {t} RECOVER PARTITIONS")
    spark.sql(f"REFRESH TABLE {t}")

# ---------------------------------------------------------------- loop
_panel()
for i_, m in enumerate(TODO):
    t_ = time.time()
    _CUR.update(m=m, t=time.time())
    rec = {"month": m}
    try:
        need_ = [mm for mm in ALL_MONTHS if month_add(m, -max(WINDOW_MONTHS, 4)) < mm <= m]
        step("node-edge caches"); ensure_ec(need_)
        if RUN_LOCAL and not local_done(m):
            prev_m = month_add(m, -1)
            if prev_m in ALL_MONTHS and not exists(st_path(prev_m) + "/_SUCCESS"):
                # warm-up: rebuild the pair state over the 3 months before m (tie age then counts from there)
                for wm_ in [mm for mm in ALL_MONTHS if month_add(m, -4) < mm < m]:
                    step(f"warm-up state {wm_}"); ensure_ec([wm_]); build_local(wm_, write=False)
            rec["local_rows"] = build_local(m)
        if RUN_EXPORT and not export_done(m):
            step("window export"); cnt = export_window(m)
            rec.update({"int_edges": cnt["int_edges"], "obs_edges": cnt["obs_edges"], "com_edges": cnt["com_edges"], "nodes": cnt["nodes"]})
        prune(f"{WORK}/state", {month_add(m, -k) for k in range(KEEP_STATES)})
        prune(f"{WORK}/ec", set(need_[-KEEP_EC:])); prune(f"{WORK}/tot", set(need_[-KEEP_EC:]))
    except Exception as e:
        _stop_heartbeat()
        _panel(sec(f"Stopped at {m}: {str(e).splitlines()[0][:200]}. Finished months are kept: rerun this cell (or Run All) "
                   f"and it resumes at {m}."))
        raise
    rec["minutes"] = round((time.time() - t_) / 60, 2)
    LOG.append(rec)
    pd.DataFrame([{**rec, "finished_at": dt.datetime.now().isoformat(timespec="seconds")}]).to_csv(
        f"{OUT_DIR}/run_log.csv", mode="a", header=not os.path.exists(f"{OUT_DIR}/run_log.csv"), index=False)
    _CUR["m"] = None; _panel()
    if NEW_SESSION_EACH_MONTH and i_ < len(TODO) - 1:
        _CUR.update(m=TODO[i_ + 1], step="starting a fresh Spark session", t=time.time()); _panel()
        new_session()
if RUN_LOCAL:
    register_local()
if RUN_EXPORT and all(export_done(m) for m in RUN_MONTHS):
    json.dump({"sig": SIG, "windows": RUN_MONTHS, "at": dt.datetime.now().isoformat(timespec="seconds")},
              open(os.path.join(NET_DIR, "_EXPORT_FINISHED"), "w"))
_stop_heartbeat()
_panel(sec("Done: every month built and exported" if not TODO or LOG else "Nothing to build"))

## §4 Summary of this part

Rows per month and a quick look at the local metrics of the latest month (shares and counts, scope customers).

In [ ]:
# §4  Summary
lt = spark.table(OUT_TABLES["local"])
lm = max(partitions(OUT_TABLES["local"], "month"))
L = lt.where(F.col("month") == lm)
num = [c for c, t in L.dtypes if t in ("double", "bigint", "int") and c not in ("month",)]
q = L.select(*[F.expr(f"percentile_approx({c}, array(0.1, 0.5, 0.9))").alias(c) for c in num]).first()
cov = L.select(*[F.avg(F.col(c).isNotNull().cast("double")).alias(c) for c in num]).first()
QS = pd.DataFrame([{"column": c, "filled": cov[c], "p10": (q[c] or [None] * 3)[0], "p50": (q[c] or [None] * 3)[1],
                    "p90": (q[c] or [None] * 3)[2]} for c in num])
rows = lt.groupBy("month").count().orderBy("month").toPandas()
gs = pd.read_csv(f"{OUT_DIR}/graph_sizes.csv").drop_duplicates("window_end", keep="last") if os.path.exists(f"{OUT_DIR}/graph_sizes.csv") else pd.DataFrame()
out(sec(f"§4 {OUT_TABLES['local']} · latest month {lm} · {L.count():,} customers"),
    tbl(QS, pct=("filled",), dec=("p10", "p50", "p90")),
    sec("Rows per month"), tbl(rows),
    sec("Exported graphs per window"), tbl(gs) if len(gs) else "",
    para("Impala: INVALIDATE METADATA " + OUT_TABLES["local"] + ";"))